# Pretrained and neural models on the 2025-26 RTM window (Kaggle GPU)

Four models on the same 150 days, folds and scores as the CPU model comparison, one cell each:
Chronos-Bolt and TimesFM (pretrained, no training on this data) and N-HiTS and TFT (trained here).

* **Chronos-Bolt, TimesFM:** for each forecast day D the model reads the filled RTM price up to the
  end of day D - 1 and forecasts the 96 blocks of D. They see no other input.
* **N-HiTS, TFT:** trained once per fold on every earlier day, then used for the 30 days of the
  fold without retraining. Inputs: the RTM price history, and for day D the DAM price, the block
  and the weekday, which are all known when day D is forecast.

**Saving.** Each model cell writes a checkpoint after every fold to `/kaggle/working/checkpoints`
and its full predictions to `/kaggle/working/predictions_<model>.parquet` the moment it finishes.
Running a cell again skips the folds that already have a checkpoint, so a failure loses at most the
fold that was running. Download the finished files as you go. To keep them after the session ends,
use Save Version.

**Input:** a private Kaggle dataset holding `rtm_dam_prices.parquet`, written by
`scripts/export_kaggle_inputs.py`. It has RTM and DAM prices and fold labels only.

**Status:** written without a GPU. The model calls follow the public descriptions of the libraries
and have not been run yet. The first run decides whether they need changing.

In [ ]:
%pip install -q chronos-forecasting "timesfm[torch]"

In [ ]:
import shutil
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

CONTEXT = 2016  # blocks of history per forecast for the pretrained models: 21 days
HORIZON = 96
BATCH = 16
CAP = 10000.0
CAP_SHARE = 0.95
OUT = Path("/kaggle/working")
CKPT = OUT / "checkpoints"
CKPT.mkdir(parents=True, exist_ok=True)
RUN = ["chronos_bolt", "timesfm", "nhits", "tft"]
CHRONOS_ID = "amazon/chronos-bolt-base"
TIMESFM_ID = "google/timesfm-2.5-200m-pytorch"

In [ ]:
found = sorted(Path("/kaggle/input").rglob("rtm_dam_prices.parquet")) or [
    Path("rtm_dam_prices.parquet")
]
data = pd.read_parquet(found[0]).sort_values(["date", "block"]).reset_index(drop=True)
history = data["rtm_price_filled"].to_numpy(dtype=float)
first_index = data.groupby("date").head(1)
first_row = {day: int(i) for i, day in zip(first_index.index, first_index["date"], strict=True)}
fold_days = {
    int(k): sorted(data.loc[data["fold"] == k, "date"].unique())
    for k in sorted(data["fold"].dropna().unique())
}
print(found[0], len(data), "rows, folds:", {k: len(v) for k, v in fold_days.items()})

In [ ]:
def labelled(day, row):
    """One day of predictions in the format of the model comparison."""
    rows = data[data["date"] == day]
    part = rows[["date", "block", "rtm_price", "fold", "period"]].rename(
        columns={"rtm_price": "actual"}
    )
    part["prediction"] = row[part["block"].to_numpy() - 1]
    return part


def checkpointed(name, fold, compute):
    """The fold's predictions from its checkpoint, or computed and written to it at once."""
    path = CKPT / f"{name}_fold{fold}.parquet"
    if path.exists():
        print(f"{name} fold {fold}: checkpoint found, skipped")
        return pd.read_parquet(path)
    started = time.time()
    part = compute()
    part.to_parquet(path, index=False)
    print(f"{name} fold {fold}: {time.time() - started:.0f}s, saved")
    return part


def finish(name, parts):
    """Write the model's full predictions as soon as its last fold is done."""
    frame = pd.concat(parts, ignore_index=True)
    frame["model"] = name
    frame["fold"] = frame["fold"].astype(int)
    frame.to_parquet(OUT / f"predictions_{name}.parquet", index=False)
    print("wrote", OUT / f"predictions_{name}.parquet", len(frame), "rows")
    return frame


def contexts_for(batch):
    return [history[max(0, first_row[day] - CONTEXT) : first_row[day]] for day in batch]


def forecast_days(predict, name):
    """Forecast every fold day with a model that reads a context. ``predict`` takes a list of
    arrays and returns (days, 96)."""
    parts = []
    for fold, days in fold_days.items():

        def compute(days=days):
            pieces = []
            for first in range(0, len(days), BATCH):
                batch = days[first : first + BATCH]
                values = np.asarray(predict(contexts_for(batch)), dtype=float)
                if values.shape != (len(batch), HORIZON):
                    raise ValueError(
                        f"{name}: got {values.shape}, expected {(len(batch), HORIZON)}"
                    )
                pieces += [labelled(day, row) for day, row in zip(batch, values, strict=True)]
            return pd.concat(pieces, ignore_index=True)

        parts.append(checkpointed(name, fold, compute))
    return finish(name, parts)


def yesterday(contexts):
    return np.stack([c[-HORIZON:] for c in contexts])

## Check: yesterday's price, built the same way as the comparison's own

Its scores must match the `naive_yesterday` row of the comparison.

In [ ]:
forecast_days(yesterday, "naive_yesterday_check")

## Chronos-Bolt (pretrained)

In [ ]:
if "chronos_bolt" in RUN:
    try:
        import torch
        from chronos import BaseChronosPipeline

        pipeline = BaseChronosPipeline.from_pretrained(
            CHRONOS_ID,
            device_map="cuda" if torch.cuda.is_available() else "cpu",
            torch_dtype=torch.bfloat16,
        )

        def chronos_predict(contexts):
            tensors = [torch.tensor(c, dtype=torch.float32) for c in contexts]
            forecast = np.asarray(pipeline.predict(tensors, prediction_length=HORIZON).float())
            return forecast[:, forecast.shape[1] // 2, :]  # the median quantile

        forecast_days(chronos_predict, "chronos_bolt")
    except Exception:
        traceback.print_exc()

## TimesFM (pretrained)

In [ ]:
if "timesfm" in RUN:
    try:
        import timesfm

        model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(TIMESFM_ID)
        model.compile(
            timesfm.ForecastConfig(
                max_context=CONTEXT,
                max_horizon=HORIZON,
                normalize_inputs=True,
                use_continuous_quantile_head=True,
            )
        )

        def timesfm_predict(contexts):
            point, _ = model.forecast(horizon=HORIZON, inputs=contexts)
            return np.asarray(point)

        forecast_days(timesfm_predict, "timesfm")
    except Exception:
        traceback.print_exc()

## Neural models: shared setup

Installs the library in its own cell, because it may change the versions of packages the two
pretrained models already loaded. Run the cells above first.

In [ ]:
%pip install -q neuralforecast

In [ ]:
frame = data.copy()
frame["unique_id"] = "rtm"
frame["ds"] = frame["timestamp"].dt.tz_localize(None)  # local clock time, no time zone
frame["y"] = frame["rtm_price_filled"]
frame["weekday"] = frame["date"].dt.day_of_week.astype(float)
frame["block"] = frame["block"].astype(float)
EXOG = ["dam_price", "block", "weekday"]
long = frame[["unique_id", "ds", *EXOG, "y"]]
lookup = frame[["ds", "date", "rtm_price", "fold", "period"]].rename(
    columns={"rtm_price": "actual"}
)


def neural_forecasts(name, make_model):
    """Train once per fold on the days before it and forecast each of its days.

    The data given to the library ends with the last day of the fold, so its last windows of 96
    blocks are the fold's days. The model is not retrained inside the fold.
    """
    from neuralforecast import NeuralForecast

    parts = []
    for fold, days in fold_days.items():

        def compute(days=days):
            end = frame.loc[frame["date"] == days[-1], "ds"].max()
            nf = NeuralForecast(models=[make_model()], freq="15min")
            cv = nf.cross_validation(
                df=long[long["ds"] <= end], n_windows=len(days), step_size=HORIZON, refit=False
            )
            column = [c for c in cv.columns if c not in ("unique_id", "ds", "cutoff", "y")][0]
            part = cv[["ds", column]].rename(columns={column: "prediction"}).merge(lookup, on="ds")
            part["block"] = (part["ds"].dt.hour * 4 + part["ds"].dt.minute // 15 + 1).astype(int)
            return part[["date", "block", "actual", "fold", "period", "prediction"]]

        parts.append(checkpointed(name, fold, compute))
    return finish(name, parts)

## N-HiTS (trained here)

In [ ]:
if "nhits" in RUN:
    try:
        from neuralforecast.losses.pytorch import MAE
        from neuralforecast.models import NHITS

        def make_nhits():
            return NHITS(
                h=HORIZON,
                input_size=HORIZON * 7,
                futr_exog_list=EXOG,
                loss=MAE(),
                scaler_type="standard",
                max_steps=1500,
                random_seed=0,
            )

        neural_forecasts("nhits", make_nhits)
    except Exception:
        traceback.print_exc()

## TFT (trained here)

In [ ]:
if "tft" in RUN:
    try:
        from neuralforecast.losses.pytorch import MAE
        from neuralforecast.models import TFT

        def make_tft():
            return TFT(
                h=HORIZON,
                input_size=HORIZON * 7,
                futr_exog_list=EXOG,
                loss=MAE(),
                hidden_size=64,
                batch_size=32,
                scaler_type="robust",
                max_steps=1000,
                random_seed=0,
            )

        neural_forecasts("tft", make_tft)
    except Exception:
        traceback.print_exc()

## Scores

In [ ]:
def scores(frame):
    rows = {}
    for name, part in frame[frame["actual"].notna()].groupby("model"):
        a, p = part["actual"].to_numpy(), part["prediction"].to_numpy()
        at_cap, called = a >= CAP, p >= CAP_SHARE * CAP
        hits = int((at_cap & called).sum())
        prec = hits / max(int(called.sum()), 1)
        rec = hits / max(int(at_cap.sum()), 1)
        rows[name] = {
            "MAE": np.mean(np.abs(a - p)),
            "RMSE": np.sqrt(np.mean((a - p) ** 2)),
            "sMAPE": np.mean(2 * np.abs(a - p) / (np.abs(a) + np.abs(p))) * 100,
            "cap_f1": 2 * prec * rec / (prec + rec) if prec + rec > 0 else float("nan"),
        }
    return pd.DataFrame(rows).T


files = sorted(OUT.glob("predictions_*.parquet"))
everything = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print("models:", sorted(everything["model"].unique()))
for label, part in [
    ("test", everything[everything["period"] == "test"]),
    ("validation", everything[everything["period"] == "validation"]),
    *[(f"fold {k}", everything[everything["fold"] == k]) for k in fold_days],
]:
    print(f"== {label} ==")
    print(scores(part).round(2).to_string())

In [ ]:
# Bundle the finished files so they can be downloaded in one go.
shutil.make_archive(str(OUT / "predictions_bundle"), "zip", OUT, ".")
print("bundle:", OUT / "predictions_bundle.zip")